# Email Agent

Creating an agent that authenticates the user, reads emails from an inbox and drafts a response for HITL review.

#### Design

- Utilize agent state to store the user authentication, create and manage the email state
- Develop a tool that reads the email (use @wrap_model_call to base access on the user type)
- A tool that drafts an email and is interrupted for human review and approval

In [1]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware, ModelRequest, ModelResponse, wrap_model_call
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command, Callable
from langchain.messages import HumanMessage

from dataclasses import dataclass

from dotenv import load_dotenv

from typing import Dict, Any

In [2]:
load_dotenv()

True

## Setting Up Context and Tools

Next steps:
- Create tools to write passed state objects to the agent graph
- 

In [3]:
class EmailUser(AgentState):
    user_auth: str
    inbox: Dict[str, Any]
    outbox: Dict[str, Any]

In [4]:
@tool
def read_inbox(runtime: ToolRuntime, email_number: int) -> str:
    """Reads emails from the user's inbox.
       Takes an email number as an argument, which is the position
       of the email in the inbox"""

    try:
        return runtime.state["inbox"][email_number]
    except Exception as e:
        return f"Could not read inbox, error: {e}"


@wrap_model_call
def read_inbox_permission(request: ModelRequest,
                          handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Gives permission to the agent to read the user's inbox if they are authorized"""

    if user_auth == "authorized":
        pass
    else:
        tools = None
        request = request.override(tools=tools)

    return handler(request)

@tool
def send_email(runtime: ToolRuntime, body: str) -> str:
    """Scans the outbox to find the right position.
       Sends email back to the sender"""

    try:
        max_outbox_position = max(i for i in runtime.state["outbox"])
    except Exception as e:
        print(f"Cannot read outbox, error: {e}")
    
    try:
        runtime.state["outbox"][max_outbox_position + 1] = body
    except Exception as e:
        print(f"Unable to send email, exception: {e}")



## Building the Agent

In [5]:
email_agent = create_agent(
    model="claude-haiku-4-5",
    tools=[read_inbox, send_email],
    state_schema=EmailUser,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware[AgentState, None](
            interrupt_on={
                "read_inbox": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval"
        )
    ],
    system_prompt="""You are a helpful email assistant. You will read my inbox
    and automatically draft responses. Do not ask me for confirmation of the 
    written content, an approval step already exists."""
)

In [6]:
user_auth = "authorized"
inbox = {0: {"Hey, can we get some time to talk about the agent project? I am free at 2:00 p.m. CST today."},
         1: {"Please see attached for an update on project X. We are on track to deliver ahead of schedule and below budget!"}}
outbox = {0: {"This is the first message I have sent!"}}

In [7]:
config={"configurable": {"thread_id": "1"}}

response = email_agent.invoke(
    {"messages": [HumanMessage(content=f"{user_auth}, {inbox}, {outbox}")]},
    config=config
)

response

{'messages': [HumanMessage(content="authorized, {0: {'Hey, can we get some time to talk about the agent project? I am free at 2:00 p.m. CST today.'}, 1: {'Please see attached for an update on project X. We are on track to deliver ahead of schedule and below budget!'}}, {0: {'This is the first message I have sent!'}}", additional_kwargs={}, response_metadata={}, id='b7ec3805-7c8b-4e98-9558-95a8e420fad0'),
  AIMessage(content=[{'text': "I'll read these emails from your inbox and draft responses for you.", 'type': 'text'}, {'id': 'toolu_011e23dC1x78x4jaGrVugpmN', 'caller': {'type': 'direct'}, 'input': {'email_number': 0}, 'name': 'read_inbox', 'type': 'tool_use'}, {'id': 'toolu_01E2npMbMt2RCHEtYkVTTzUN', 'caller': {'type': 'direct'}, 'input': {'email_number': 1}, 'name': 'read_inbox', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CetuzapxtWUZJf7ex9kL3', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 

In [9]:
response = email_agent.invoke(
    Command[tuple[()]](
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

response

{'messages': [HumanMessage(content="authorized, {0: {'Hey, can we get some time to talk about the agent project? I am free at 2:00 p.m. CST today.'}, 1: {'Please see attached for an update on project X. We are on track to deliver ahead of schedule and below budget!'}}, {0: {'This is the first message I have sent!'}}", additional_kwargs={}, response_metadata={}, id='b7ec3805-7c8b-4e98-9558-95a8e420fad0'),
  AIMessage(content=[{'text': "I'll read these emails from your inbox and draft responses for you.", 'type': 'text'}, {'id': 'toolu_011e23dC1x78x4jaGrVugpmN', 'caller': {'type': 'direct'}, 'input': {'email_number': 0}, 'name': 'read_inbox', 'type': 'tool_use'}, {'id': 'toolu_01E2npMbMt2RCHEtYkVTTzUN', 'caller': {'type': 'direct'}, 'input': {'email_number': 1}, 'name': 'read_inbox', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CetuzapxtWUZJf7ex9kL3', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 